# Explanation Notebook for ScenarioModel8
*Model Type:* **RandomForest**
*User:* UserAutoAdv (Expertise: Advanced, Formats: [visual], Details: high, Purpose: general)
*Explanation Method:* PDP *(auto-selected)*


## Training the RandomForest Model
We train a **RandomForest** model on the provided dataset.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
!pip install -q scikit-learn pandas matplotlib
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv(r"data/breast_cancer.csv")
y = df['target']
X = df.drop(columns=['target']).values
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    print('[Info] Used stratified split to preserve class proportions in train/test.')
except Exception:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    print('[Info] Used non-stratified split (stratification not applicable).')
class Dummy: pass
data = Dummy(); data.feature_names = list(df.drop(columns=['target']).columns)
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=0)
print('[Info] Normalization disabled by training policy.')
model.fit(X_train, y_train)
print('Accuracy: RandomForest = ' + format(model.score(X_test, y_test), '.2f'))


## Explaining the model using PDP
Partial Dependence Plot (PDP) shows how a feature affects the model's predictions on average.


In [ ]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt

def _get_feature_names():
    try:
        names = list(getattr(data, 'feature_names', []))
        if names and len(names) == X_train.shape[1]:
            return names
    except Exception:
        pass
    if 'df' in globals():
        cols = [c for c in df.columns if c.lower() not in ('label','target','y')]
        if len(cols) == X_train.shape[1]:
            return cols
    return [f'f{j}' for j in range(X_train.shape[1])]

RAW_NAMES = _get_feature_names()
def _humanize(name):
    s = name.replace('_',' ').strip()
    s = re.sub(r'(?<!^)(?=[A-Z])',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return ' '.join([w.upper() if w.lower() in {'svm','pdp','ice','api','url'} else w.capitalize() for w in s.split(' ')])
HUMAN = [_humanize(n) for n in RAW_NAMES]

max_display = 10
table_rows = 14
curve_resolution = 40
ice_instances = 20
metric_digits = 6
figure_height = 5.4
line_width = 2.6
pdp_sample_rows = 10
ice_detailed_instances = 5
sampled_curve_points = 10
print('\n[PDP overview]')
print('Partial dependence shows the average effect of a feature on the model prediction.')
feat_idx = 0
xs = X_train[:, feat_idx]
grid = np.linspace(np.min(xs), np.max(xs), num=curve_resolution)
preds = []
for g in grid:
    X_tmp = X_train.copy()
    X_tmp[:, feat_idx] = g
    if hasattr(model, 'predict_proba'):
        preds.append(float(np.mean(model.predict_proba(X_tmp)[:,1])))
    else:
        preds.append(float(np.mean(model.predict(X_tmp))))
df_pdp = pd.DataFrame({'Feature':[HUMAN[feat_idx]]*len(grid), 'Value':grid, 'AveragePrediction':preds})
start_pred = float(df_pdp['AveragePrediction'].iloc[0])
end_pred = float(df_pdp['AveragePrediction'].iloc[-1])
delta_pred = end_pred - start_pred
if delta_pred > 0.01:
    trend_label = 'overall increasing'
elif delta_pred < -0.01:
    trend_label = 'overall decreasing'
else:
    trend_label = 'mostly stable'
sample_idx = np.unique(np.linspace(0, len(df_pdp)-1, num=min(pdp_sample_rows, len(df_pdp))).astype(int))
df_pdp_view = df_pdp.iloc[sample_idx].copy()
df_pdp_view['DeltaFromStart'] = df_pdp_view['AveragePrediction'] - start_pred
print('\n[Visual explanation]')
print('How to read the plot: PDP is a global explanation. It averages over the remaining features to isolate the marginal effect of the selected feature.')
print('Look for increasing, decreasing, or flat regions to understand how the model response changes across the feature range.')
plt.figure(figsize=(8, figure_height))
plt.plot(df_pdp['Value'], df_pdp['AveragePrediction'], linewidth=line_width)
plt.title('Partial dependence')
plt.xlabel(HUMAN[feat_idx])
plt.ylabel('Average prediction')
plt.grid(alpha=0.2)
plt.scatter([df_pdp['Value'].iloc[0], df_pdp['Value'].iloc[len(df_pdp)-1]],[start_pred, end_pred])
min_idx = int(np.argmin(df_pdp['AveragePrediction'].values))
max_idx = int(np.argmax(df_pdp['AveragePrediction'].values))
plt.scatter([df_pdp['Value'].iloc[min_idx], df_pdp['Value'].iloc[max_idx]],[df_pdp['AveragePrediction'].iloc[min_idx], df_pdp['AveragePrediction'].iloc[max_idx]])
baseline = float(np.mean(df_pdp['AveragePrediction'].values))
plt.axhline(baseline, linestyle='--', linewidth=1.0)
plt.tight_layout(); plt.show()
